# Experimentos con la Gramatica Lark

**Modulo:** Exploratory  
**Objetivo:** Explorar la gramatica del parser de pseudocodigo  
**Duracion estimada:** 30 minutos

---

## Contenido

1. [Setup](#setup)
2. [Introduccion a Lark](#introduccion-a-lark)
3. [Estructura de la Gramatica](#estructura-de-la-gramatica)
4. [Reglas Principales](#reglas-principales)
5. [Terminales y Tokens](#terminales-y-tokens)
6. [Prioridades y Ambiguedad](#prioridades-y-ambiguedad)
7. [Ejercicios](#ejercicios)

---

## 1. Setup

In [ ]:
# Agregar path del proyecto
import sys
sys.path.insert(0, '../..')

# Imports necesarios
from app.core.parser import parse_pseudocode
from app.core.parser.pseudocode_parser import PseudocodeParser

# Crear instancia del parser
parser = PseudocodeParser()

print("Setup completado")

---

## 2. Introduccion a Lark

Lark es una libreria de parsing para Python que soporta gramaticas EBNF.
El proyecto utiliza Lark por las siguientes ventajas:

### Caracteristicas Principales

| Caracteristica | Descripcion |
|----------------|-------------|
| Parser LALR | Parsing eficiente O(n) |
| Propagacion de posicion | Linea y columna en cada token |
| Transformers | Conversion automatica a AST |
| Soporte Unicode | Identificadores en espanol |

### Archivo de Gramatica

La gramatica se encuentra en:

```
app/core/parser/grammar/pseudocode.lark
```

### Ejemplo: Parsing Basico

In [ ]:
# Parsing basico: ver el proceso
codigo = """
algorithm ejemplo(n)
begin
    x <- 1
end
"""

# Parsear y obtener el AST
ast = parser.parse(codigo)

print("=== RESULTADO DEL PARSING ===")
print(f"Tipo resultado: {type(ast).__name__}")
print(f"Algoritmo: {ast.algorithm.name}")
print(f"Parsing exitoso!")

---

## 3. Estructura de la Gramatica

La gramatica esta organizada en secciones logicas:

### 1. Programa Principal

```lark
?start: program
program: class_definition* algorithm
```

### 2. Definicion de Algoritmo

```lark
algorithm: ALGORITHM_KW IDENTIFIER "(" parameter_list? ")" block
```

### 3. Bloque de Codigo

```lark
block: BEGIN statement_list END
```

### Ejemplo: Probar Diferentes Estructuras

In [ ]:
# Probar diferentes estructuras de codigo
estructuras = [
    ("Asignacion simple", """
algorithm test(n)
begin
    x <- 5
end
"""),
    ("Ciclo For", """
algorithm test(n)
begin
    for i <- 1 to n do
        x <- i
    end
end
"""),
    ("Ciclo While", """
algorithm test(n)
begin
    while (n > 0) do
        n <- n - 1
    end
end
"""),
    ("Condicional If-Else", """
algorithm test(n)
begin
    if (n > 0) then
        x <- 1
    else
        x <- 0
    end
end
""")
]

print("=== PARSING DE ESTRUCTURAS ===")
for nombre, codigo in estructuras:
    try:
        ast = parser.parse(codigo)
        stmt_type = type(ast.algorithm.body.statements[0]).__name__
        print(f"{nombre}: OK -> {stmt_type}")
    except Exception as e:
        print(f"{nombre}: ERROR -> {e}")

---

## 4. Reglas Principales

### Asignacion

El operador de asignacion (`ASSIGN`) acepta:
- Flecha izquierda: `<-`
- Dos puntos igual: `:=`

### Ciclos

- **For**: `for i <- 1 to n do ... end`
- **While**: `while (condicion) do ... end`
- **Repeat**: `repeat ... until (condicion)`

### Ejemplo: Operadores de Asignacion

In [ ]:
# Probar diferentes operadores de asignacion
codigo_flecha = """
algorithm test(n)
begin
    x <- 5
end
"""

codigo_dos_puntos = """
algorithm test(n)
begin
    x := 5
end
"""

print("=== OPERADORES DE ASIGNACION ===")
for nombre, codigo in [("Flecha (<-)", codigo_flecha), ("Dos puntos (:=)", codigo_dos_puntos)]:
    try:
        ast = parser.parse(codigo)
        print(f"{nombre}: Parsing exitoso")
    except Exception as e:
        print(f"{nombre}: Error - {e}")

---

## 5. Terminales y Tokens

### Palabras Clave (Bilingues)

| Ingles | Espanol |
|--------|---------|
| algorithm | algoritmo |
| begin | inicio |
| end | fin |
| for | para |
| while | mientras |
| if | si |
| return | retornar |

### Ejemplo: Parsing Bilingue

In [ ]:
# Probar sintaxis en ingles y espanol
codigo_ingles = """
algorithm sum(n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + i
    end
    return total
end
"""

codigo_espanol = """
algoritmo suma(n)
inicio
    total <- 0
    para i <- 1 hasta n hacer
        total <- total + i
    fin
    retornar total
fin
"""

print("=== PARSING BILINGUE ===")
for nombre, codigo in [("Ingles", codigo_ingles), ("Espanol", codigo_espanol)]:
    try:
        ast = parser.parse(codigo)
        print(f"{nombre}:")
        print(f"  Algoritmo: {ast.algorithm.name}")
        print(f"  Statements: {len(ast.algorithm.body.statements)}")
    except Exception as e:
        print(f"{nombre}: Error - {e}")

---

## 6. Prioridades y Ambiguedad

### Precedencia de Operadores

Los operadores tienen la siguiente precedencia (de menor a mayor):
1. OR (logico)
2. AND (logico)
3. Comparacion (<, >, =, etc.)
4. Suma, Resta
5. Multiplicacion, Division

### Ejemplo: Expresiones Complejas

In [ ]:
# Probar expresiones complejas
expresiones = [
    "x <- a + b * c",        # b*c primero
    "x <- (a + b) * c",      # a+b primero
    "x <- a > b and c < d",  # comparaciones primero, luego and
    "x <- floor((low + high) / 2)",  # funcion floor
]

print("=== EXPRESIONES COMPLEJAS ===")
for expr in expresiones:
    codigo = f"""
algorithm test(a, b, c, d, low, high)
begin
    {expr}
end
"""
    try:
        ast = parser.parse(codigo)
        print(f"OK: {expr}")
    except Exception as e:
        print(f"ERROR: {expr} -> {e}")

---

## 7. Ejercicios

### Ejercicio 1: Crear y Parsear un Algoritmo

In [ ]:
# Ejercicio 1: Escribe un algoritmo con ciclo while y condicional
# Luego parsealo y verifica que funcione

tu_codigo = """
algorithm miAlgoritmo(n)
begin
    i <- 0
    while (i < n) do
        if (i > 5) then
            x <- i * 2
        end
        i <- i + 1
    end
end
"""

# Parsear tu codigo
ast = parser.parse(tu_codigo)
print(f"Algoritmo: {ast.algorithm.name}")
print(f"Numero de statements: {len(ast.algorithm.body.statements)}")

### Ejercicio 2: Probar Manejo de Errores

In [ ]:
# Ejercicio 2: Que pasa con codigo mal formado?
codigos_incorrectos = [
    ("Falta end", """
algorithm test(n)
begin
    x <- 5
"""),
    ("Falta begin", """
algorithm test(n)
    x <- 5
end
"""),
    ("Sintaxis incorrecta", """
algorithm test(n)
begin
    for i = 1 to n  # Usar = en lugar de <-
        x <- i
    end
end
""")
]

print("=== MANEJO DE ERRORES ===")
for nombre, codigo in codigos_incorrectos:
    try:
        ast = parser.parse(codigo)
        print(f"{nombre}: Inesperadamente exitoso")
    except Exception as e:
        print(f"{nombre}: Error capturado correctamente")

---

## 8. Tips y Resumen

### Tips

- La gramatica soporta sintaxis en ingles y espanol
- Usa `<-` o `:=` para asignacion (evitar `=`)
- Los comentarios con `//` o `->>` son ignorados
- El parser es case-insensitive para palabras clave

### Resumen

Has aprendido:
- Como funciona el parser Lark con gramatica EBNF
- Las estructuras principales (algoritmo, ciclos, condicionales)
- Soporte bilingue para palabras clave
- Precedencia de operadores
- Manejo de errores de sintaxis

---

## Proximos Pasos

- **ast_visualization.ipynb**: Visualizar el AST generado
- **parser_testing.ipynb**: Mas pruebas de parsing

**Siguiente modulo recomendado**: `02_complexity_analysis/` para analizar complejidad